# SESTRAV - Canonical GNN v2.3 (mean-pool) retrain on GPU

> **HISTORICAL ARTIFACT - READ BEFORE USING ANY NUMBER BELOW.** This notebook
> regenerates a v4-era checkpoint against the v4 corpus and
> `models/peptide_binding_matrix_v4.csv`. It has never been executed (every cell
> shows `execution_count: null`). The v5 peptide-grouped GNN run of 2026-08-13
> supersedes it: Gate 1 pooled AUC-PR 0.6458 against a threshold of 0.65, FAIL by
> 0.0042, and +0.0402 over the RF mode-31 baseline. Numbers in this notebook are v4
> and are NOT comparable to any v5 figure. Note also that
> `models/peptide_binding_matrix_v4.csv` covers only 0.79% of the v5 corpus's
> tested-negative rows, so using it against a v5 corpus produces a label proxy worth
> roughly -0.154 pooled AUC-PR; it is correct here only because the corpus is also v4.

**Purpose.** Regenerate a v4-era repo-integrity artifact: the canonical untagged
`models/gnn/structural_gnn_v2.pth` + scalers + `gnn_config.json` + OOF (the v2.3 mean-pool
checkpoint). The local dev box is CPU-only torch, so this is offloaded to a Colab GPU.

**What this reproduces:** GINEConv + ESM-2 t12 (35M, 480-dim node features) + mode-31 physico/binding
features, mean readout, 5-fold OOF + Platt calibration, then a final all-data refit. The pooled
OOF AUC-PR that reproduces from the original run's tracked artifact is **0.7160** (v4 corpus, v4
binding matrix, ungrouped splitter carrying the D15 exact-peptide leakage). The mean-fold figure
once reported from that same run is RETRACTED as unreproducible; see `README.md` and
`ARCHITECTURE.md`.

---
### Inputs and the gitignored-data problem (read first)
A bare `git clone` does **not** contain everything. Status of the three inputs:

| Input | Size | In public repo? | How this notebook gets it |
|---|---|---|---|
| `models/peptide_binding_matrix_v4.csv` | 2.5 MB | yes (tracked) | from the clone |
| `data/immunogenicity_dataset_v4.csv` | 1.7 MB | **no (gitignored)** | you upload it (Step 3) |
| `data/esm2_embeddings_t12.pt` | 252 MB | **no (gitignored)** | regenerated on GPU (Step 4) |

### Reproducibility caveat
The environment that produced the original run is no longer available, so a different GPU or package
build will land at a different number. The GNN is a research track, not a promoted scorer: the v5
peptide-grouped comparator is RF mode-31, pooled AUC-PR **0.6055** against
`models/peptide_binding_matrix_v5.csv` (`results/pooled_cv_metrics_mode31.csv`). The v4-era 0.7635
sometimes quoted as "the production scorer" is bound to no tracked artifact
(`docs/data_registry.md`) and is not comparable to any v5 result. The stance is: pin the
numerically-relevant packages, keep `seed 42`, and record whatever number this run produces together
with its corpus, binding matrix and splitter. **No tolerance band is asserted**, because no sourced
target exists for this configuration.

**Runtime:** ~5-40 min on a T4 (embedding regen + 5-fold train + final refit). Set
`Runtime > Change runtime type > GPU` first.

## Step 1 - Confirm GPU

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Set Runtime > Change runtime type > GPU (T4 or better), then re-run.")

## Step 2 - Clone the repo and install pinned dependencies

Numerically-relevant pins come from `environments/requirements.lock` (scikit-learn 1.8.0,
torch-geometric 2.7.0). `torch` is left as Colab's CUDA build by design - the lock pins the CPU
build 2.13.0, and the GPU build is what we actually want here. If pip changes the numpy/torch ABI and
asks you to restart, use `Runtime > Restart session` and re-run from this cell.

In [ ]:
import os

REPO = "/content/SESTRAV"
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/Gavin-Borges/SESTRAV.git {REPO}
%cd {REPO}
!pip -q install 'torch-geometric==2.7.0' 'scikit-learn==1.8.0' 'transformers>=4.40,<5' joblib pyyaml
print("\nInstall complete. Optional byte-fidelity pins (may force a runtime restart):")
print("  numpy==2.4.6  scipy==1.17.1  pandas==3.0.3")

## Step 3 - Provide the v4 dataset (gitignored, 1.7 MB)

Upload `data/immunogenicity_dataset_v4.csv` from your machine. (Alternative: mount Google Drive and
copy it - commented below.) Expected: ~14,699 rows, ~45.5% positive.

In [ ]:
import os
import pandas as pd

DATA = "data/immunogenicity_dataset_v4.csv"
os.makedirs("data", exist_ok=True)
if not os.path.exists(DATA):
    from google.colab import files

    print("Select immunogenicity_dataset_v4.csv to upload ...")
    up = files.upload()
    name = list(up.keys())[0]
    if name != DATA:
        os.replace(name, DATA)
# Alternative - Google Drive:
#   from google.colab import drive; drive.mount('/content/drive')
#   import shutil; shutil.copy2('/content/drive/MyDrive/SESTRAV/immunogenicity_dataset_v4.csv', DATA)
df = pd.read_csv(DATA)
assert {"peptide", "label"}.issubset(df.columns), "unexpected schema: " + str(list(df.columns))
print(
    "rows:",
    len(df),
    "| positive rate: {:.1%}".format(df["label"].mean()),
    "| unique peptides:",
    df["peptide"].nunique(),
)
assert 14000 <= len(df) <= 15500, "row count off - is this the v4 dataset?"

## Step 4 - Regenerate the ESM-2 t12 cache on GPU (252 MB)

This mirrors `scripts/precompute_esm2_embeddings.py` exactly (MAX_LEN=11, residue positions 1..L of
`last_hidden_state`, zero-padded to (11, 480)) but runs the forward pass on CUDA for speed and for
closer fidelity to the GPU-built original. If you already have the 252 MB `.pt`, copy it from Drive
into `data/esm2_embeddings_t12.pt` and this cell will skip regeneration.

In [ ]:
import os
import torch
import pandas as pd

CACHE = "data/esm2_embeddings_t12.pt"
MODEL = "facebook/esm2_t12_35M_UR50D"
MAX_LEN, ESM_DIM = 11, 480
if os.path.exists(CACHE):
    print("cache already present:", CACHE)
else:
    from transformers import EsmModel, EsmTokenizer

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tok = EsmTokenizer.from_pretrained(MODEL)
    mdl = EsmModel.from_pretrained(MODEL).to(dev).eval()
    peps = sorted(pd.read_csv("data/immunogenicity_dataset_v4.csv")["peptide"].unique().tolist())
    print("embedding", len(peps), "unique peptides on", dev, "...")
    emb, bs = {}, 64
    for i in range(0, len(peps), bs):
        batch = peps[i : i + bs]
        inp = tok(batch, return_tensors="pt", padding=True, truncation=True).to(dev)
        with torch.no_grad():
            out = mdl(**inp).last_hidden_state
        for j, seq in enumerate(batch):
            L = len(seq)
            res = out[j, 1 : 1 + L, :].float().cpu()
            pad = torch.zeros(MAX_LEN, ESM_DIM)
            take = min(L, MAX_LEN)
            pad[:take, :] = res[:take, :]
            emb[seq] = pad
        if (i // bs) % 20 == 0:
            print("  {}/{}".format(min(i + bs, len(peps)), len(peps)))
    torch.save(emb, CACHE)
    print("saved", len(emb), "embeddings ->", CACHE)

In [ ]:
# OPTIONAL (Step 4b) - persist the 252 MB ESM-2 t12 cache to Google Drive so your v5 depth-expansion
# work can reuse it without regenerating (the slow step), and so it survives a runtime crash.
# Uncomment to use.
# from google.colab import drive; drive.mount('/content/drive')
# import os, shutil
# os.makedirs('/content/drive/MyDrive/SESTRAV', exist_ok=True)
# shutil.copy2('data/esm2_embeddings_t12.pt', '/content/drive/MyDrive/SESTRAV/esm2_embeddings_t12.pt')
# print('cache copied to Drive')

## Step 5 - Run the canonical retrain

Canonical v2.3 mean-pool command. The clobber-fix means a mean-pool run
writes BOTH the canonical untagged paths and `*_mean`-tagged paths, so this is self-protecting.

In [ ]:
# tee captures the per-fold AUC-PR + final OOF into a log file that is bundled into the zip,
# so STATE.md records the ACTUAL regenerated number rather than a transcribed one.
!mkdir -p models/gnn && python -m src.train_gnn --architecture v2 --data data/immunogenicity_dataset_v4.csv --esm2-cache data/esm2_embeddings_t12.pt --node-dim 480 --esm2-model facebook/esm2_t12_35M_UR50D --feature-mode 31 --binding-matrix models/peptide_binding_matrix_v4.csv --pooling mean --epochs 60 --patience 10 --seed 42 2>&1 | tee training_log_gnn_v23.txt

## Step 6 - Verify (checkpoint loads against config; record the OOF AUC-PR)

In [ ]:
import json
import sys
import torch
import pandas as pd
from sklearn.metrics import average_precision_score

sys.path.insert(0, "/content/SESTRAV")
cfg = json.load(open("models/gnn/gnn_config.json"))
print("gnn_config.json:", cfg)
assert cfg["pooling"] == "mean", "pooling must be mean"
assert cfg["node_dim"] == 480, "node_dim must be 480"
assert cfg["num_continuous_features"] == 31, "num_continuous_features must be 31 (mode 31)"
from src.gnn.models import GraphPredictorV2

# Instantiate from the config's own values (source of truth) so the strict load below
# is a genuine architecture-vs-checkpoint match check, not a hardcoded guess.
m = GraphPredictorV2(
    num_continuous_features=cfg["num_continuous_features"],
    node_dim=cfg["node_dim"],
    pooling=cfg["pooling"],
)
m.load_state_dict(torch.load("models/gnn/structural_gnn_v2.pth", map_location="cpu"))
print("OK: checkpoint loads cleanly (strict) against its own config.")
oof = pd.read_csv("models/gnn_oof_predictions.csv")
ap = average_precision_score(oof["label"], oof["gnn_oof_score"])
print("OOF AUC-PR: {:.4f}".format(ap))
print(
    "Record this number with its inputs: v4 corpus, v4 binding matrix, ungrouped "
    "splitter (D15 exact-peptide leakage). It is not comparable to any peptide-grouped "
    "figure. No PASS band is asserted here because no sourced target exists for this "
    "configuration."
)

In [ ]:
# Optional: full 5-gate report. The Gate 1 threshold is >= 0.65
# (src/verify/promote_gnn.py, GATE1_AUC_PR_MIN, re-anchored 2026-08-10). This v4
# artifact fails Gate 1 and Gate 2 by PRECONDITION rather than on score: it carries
# neither a splitter nor a fold column. promote_gnn will NOT mutate config.yaml.
!python -m src.verify.promote_gnn || echo 'promote_gnn reported a gate failure (expected: Gates 1 and 2 fail by precondition - no splitter or fold column).'

## Step 7 - Package the artifacts and download

In [ ]:
import hashlib
import os
import zipfile

arts = [
    "models/gnn/structural_gnn_v2.pth",
    "models/gnn/structural_gnn_v2_mean.pth",
    "models/gnn/gnn_scaler.joblib",
    "models/gnn/gnn_scaler_mean.joblib",
    "models/gnn/gnn_platt_scaler.joblib",
    "models/gnn/gnn_platt_scaler_mean.joblib",
    "models/gnn/gnn_config.json",
    "models/gnn/gnn_config_mean.json",
    "models/gnn_oof_predictions.csv",
    "models/gnn_oof_predictions_mean.csv",
    "training_log_gnn_v23.txt",
]
zpath = "sestrav_gnn_v23_canonical_artifacts.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for a in arts:
        if os.path.exists(a):
            z.write(a)
            digest = hashlib.sha256(open(a, "rb").read()).hexdigest()[:16]
            print("{:48s} {}".format(a, digest))
        else:
            print("MISSING (not written):", a)
print("\nzip ->", zpath)
# OPTIONAL - also copy the zip to Google Drive (uncomment):
#   from google.colab import drive; drive.mount('/content/drive')
#   import shutil, os as _os
#   _os.makedirs('/content/drive/MyDrive/SESTRAV', exist_ok=True)
#   shutil.copy2(zpath, '/content/drive/MyDrive/SESTRAV/' + zpath)
from google.colab import files

files.download(zpath)

## Step 8 - On your local machine, after download

1. Unzip into the repo root so files land back at `models/gnn/` and `models/`.
2. Re-verify locally (CPU is fine for loading):
   ```bash
   python -m src.verify.promote_gnn
   ```
   Confirm `gnn_config.json` shows `pooling: mean` and record the OOF AUC-PR.
3. The canonical `structural_gnn_v2.pth` is the P0 integrity target - follow the repo's policy on
   committing model binaries.
4. Update `STATE.md`: mark the P0 GNN canonical retrain CLOSED and record the **actual** OOF AUC-PR
   produced here, naming its corpus, binding matrix and splitter.